# Throwing and Exception Contracts

CSC-239 · Module 7 · Lesson 2 of 4

You can now follow and catch a failure. This lesson makes failure part of a method’s stated behavior: decide what invalid input means, raise an appropriate exception, and preserve useful details for the caller.

Select the **Java** kernel in your Workspace. Start with a fresh kernel and run cells in order. This notebook creates its own starting state.


## Learning Goals

- Create a checked exception and either handle it or declare its propagation.
- Validate parsed input and preserve a lower-level cause when reporting a clearer application failure.


## Why This Matters

A quantity form accepts text. A nonnumeric value and a negative whole number both fail validation, but they deserve different explanations.


## Check Your Starting Point

Trace a try/catch around a method call. Recall extends, a constructor that calls super, a null reference, and a method’s input/result contract.

**My explanation:**


## Concept

### State the exception contract

A **checked exception** is an exception outside the RuntimeException and Error families. For these types, the Java compiler generally requires code that may pass the exception outward to catch it or declare it. The declaration becomes part of the method's failure contract.

An **unchecked exception** belongs to the RuntimeException or Error family. The compiler does not require its possible propagation to appear in a throws declaration. ArithmeticException and IllegalArgumentException are common RuntimeException subtypes.

Checked does not mean harmless or always recoverable. Unchecked does not mean safe to ignore. These terms describe compiler rules. Choose how to handle a failure based on the operation and its callers. Normal application code should not broadly catch Error just to keep going.

### Raise an exception now

A **throw statement** raises a particular exception object at that moment. Construct the object with new and provide a useful message:

```java
class GroupRules {
    public static int requirePositive(int groups) {
        if (groups < 1) {
            throw new IllegalArgumentException("Groups must be positive.");
        }
        return groups;
    }
}
try {
    System.out.println(GroupRules.requirePositive(0));
} catch (IllegalArgumentException problem) {
    System.out.println(problem.getMessage());
}
```

This prints Groups must be positive. The throw leaves the method before its return statement. It does not merely print a message. The caller decides what response to present.

### Declare a failure that may escape

A **throws declaration** names exception types that may leave a method and reach its caller. It does not raise an exception by itself.

A **custom exception class** gives an application failure a specific name. Extend Exception for the checked type in this example. Its constructor passes a message and cause to the superclass constructor:

```java
class InvalidCountException extends Exception {
    public InvalidCountException(String message, Throwable cause) {
        super(message, cause);
    }
}
class CountRules {
    public static int read(int count) throws InvalidCountException {
        if (count < 0) {
            throw new InvalidCountException("Count cannot be negative.", null);
        }
        return count;
    }
}
try {
    System.out.println(CountRules.read(-2));
} catch (InvalidCountException problem) {
    System.out.println(problem.getMessage());
}
```

The output is Count cannot be negative. Here Exception supplies the message and cause behavior; the subclass gives the failure its application-specific name. Throwable is the common superclass used to refer to a failure object. The null cause means there is no earlier exception attached.

Compare the two keywords: throw raises this object now; throws states a possible escaping failure in the method header. A caller in a normal Java source file must catch this checked type or add a suitable throws declaration to its own method. Adding throws does not fix the invalid input; it transfers the handling responsibility outward.

IJava accepts some top-level checked calls that a conventional Java method could not leave undeclared. Our examples keep explicit try/catch or throws so the failure contract also works in Java source files. Treat an omitted-handler example as a compiler exercise, not as a portable pattern merely because a notebook accepts it.

### Preserve the earlier cause

An **exception cause** is an earlier failure stored inside a later exception. This lets you present a clear application message while retaining the original technical detail.

Integer.parseInt converts decimal integer text to int. It accepts text such as 3 or -1, but rejects words, surrounding whitespace, and values outside the int range with NumberFormatException. NumberFormatException is an unchecked RuntimeException subtype, through IllegalArgumentException. Conversion success does not prove that the number meets your application's allowed range.

```java
class InvalidCountException extends Exception {
    public InvalidCountException(String message, Throwable cause) {
        super(message, cause);
    }
}
class CountParser {
    public static int read(String text) throws InvalidCountException {
        try {
            return Integer.parseInt(text);
        } catch (NumberFormatException cause) {
            throw new InvalidCountException("Count must be a whole number.", cause);
        }
    }
}
try {
    CountParser.read("many");
} catch (InvalidCountException problem) {
    System.out.println(problem.getMessage());
    System.out.println("Has cause: " + (problem.getCause() != null));
}
```

This prints Count must be a whole number. and Has cause: true. The catch variable cause refers to the original NumberFormatException. Passing that same object to the new exception preserves it. The getCause method returns the attached failure, or null when no cause is attached. Check for null before calling a method through that reference.

Do not replace the earlier failure with only a generic message when the original cause would help diagnose it. Also avoid wrapping a checked exception in an unchecked type simply to silence the compiler. Choose a clear caller contract and meet it.

### Separate conversion from validation

A quantity of -1 parses successfully as an int. Our worked example rejects it afterward because its stated input rule allows zero or more. Its custom exception has no conversion cause because conversion succeeded.

The independent task uses a different rule: a seat count must be at least one. The value zero therefore takes a different path from the quantity example. Read the input contract before reusing the same guard.


## Video Demonstration

Follow conversion first and the range check second. Predict the successful value, each application message, and whether an original parsing exception is attached.

<video controls preload="metadata" width="960">
  <source src="media/02_throwing_and_exception_contracts/demo.mp4" type="video/mp4">
  <track kind="captions" src="media/02_throwing_and_exception_contracts/captions.vtt" srclang="en" label="English">
  Your browser does not support embedded video.
</video>

[Read the throwing and exception contracts demonstration transcript](media/02_throwing_and_exception_contracts/transcript.md).


## Worked Example

**Subgoal 1: name the failure.** InvalidQuantityException extends Exception and retains a message and cause.

**Subgoal 2: enforce the method contract.** QuantityParser.read wraps a parsing failure and separately rejects a negative value.

**Subgoal 3: let the caller respond.** Each input has a specific catch that prints the application message and whether a cause exists.


In [ ]:
class InvalidQuantityException extends Exception {
    public InvalidQuantityException(String message, Throwable cause) {
        super(message, cause);
    }
}
class QuantityParser {
    public static int read(String text) throws InvalidQuantityException {
        int quantity;
        try {
            quantity = Integer.parseInt(text);
        } catch (NumberFormatException cause) {
            throw new InvalidQuantityException("Quantity must be a whole number.", cause);
        }
        if (quantity < 0) {
            throw new InvalidQuantityException("Quantity cannot be negative.", null);
        }
        return quantity;
    }
}
String[] inputs = {"3", "two", "-1"};
for (String input : inputs) {
    try {
        System.out.println("Quantity: " + QuantityParser.read(input));
    } catch (InvalidQuantityException problem) {
        System.out.println("Problem: " + problem.getMessage());
        System.out.println("Has cause: " + (problem.getCause() != null));
    }
}


Expected output:

```text
Quantity: 3
Problem: Quantity must be a whole number.
Has cause: true
Problem: Quantity cannot be negative.
Has cause: false
```

The text 3 converts and meets the nonnegative rule. The word two cannot convert, so the new exception retains NumberFormatException as its cause. The text -1 converts, then fails the range rule; this failure has no earlier conversion exception attached.


## Predict, Run, Trace, and Explain

### Predict conversion, validation and causes

Read the complete program before running it. Predict every output line for inputs {"8", "oops", "-3"}. For each text, decide whether conversion succeeds, whether the nonnegative rule passes, and whether a cause is attached if the method fails. Identify the statement that raises the application exception on each failed path. Record your prediction before running the next cell or opening the answer.

My predicted complete output:

For each input: conversion result, range result and attached cause:

The throw statement reached on each failed path:

Why throws does not force the successful input to fail:


In [ ]:
class InvalidQuantityException extends Exception {
    public InvalidQuantityException(String message, Throwable cause) {
        super(message, cause);
    }
}
class QuantityParser {
    public static int read(String text) throws InvalidQuantityException {
        int quantity;
        try {
            quantity = Integer.parseInt(text);
        } catch (NumberFormatException cause) {
            throw new InvalidQuantityException("Quantity must be a whole number.", cause);
        }
        if (quantity < 0) {
            throw new InvalidQuantityException("Quantity cannot be negative.", null);
        }
        return quantity;
    }
}
String[] inputs = {"8", "oops", "-3"};
for (String input : inputs) {
    try {
        System.out.println("Quantity: " + QuantityParser.read(input));
    } catch (InvalidQuantityException problem) {
        System.out.println("Problem: " + problem.getMessage());
        System.out.println("Has cause: " + (problem.getCause() != null));
    }
}


Run the whole cell once. Keep your original prediction and compare each output line. Explain the first difference using the conversion, range check, throw statement or cause argument. Retain your corrected explanation. Each attempt should run the full program so its exception class, parser and caller are present.

My original prediction:

My actual complete output:

The first difference and its cause:

Why false can appear while the caller is handling an exception:

My corrected post-run explanation:

### Trace the exception contract through callers

Complete the table for the prediction program. Name the exception type first raised on each failed path, the type received by the outer catch, and the object passed as the cause. Classify NumberFormatException and InvalidQuantityException as checked or unchecked using their class families. Explain what the compiler requires from a conventional Java method that lets the checked type reach its caller. Do those categories tell you that a failure is harmless or safe to ignore? Then use the helper programs below to trace a declared failure through another method.

| Input | Conversion result or first failure | Range check reached? | Type caught by caller | Attached cause or none |
|---|---|---|---|---|
| "8" | | | | |
| "oops" | | | | |
| "-3" | | | | |


NumberFormatException: checked or unchecked, and why:

InvalidQuantityException: checked or unchecked, and why:

What a conventional caller must catch or declare:

Why these categories do not determine how serious a failure is:

My post-run explanation:

<details>
<summary>Show answer</summary>

The text "8" converts to 8 and passes the nonnegative rule, so Quantity: 8 prints. "oops" cannot convert: Integer.parseInt raises NumberFormatException. The inner catch constructs and throws InvalidQuantityException with the application message and that original object as its cause. The outer catch prints the message and Has cause: true. "-3" converts successfully, then fails the quantity < 0 guard. The new InvalidQuantityException has a null cause because there was no earlier conversion failure; the outer catch prints Has cause: false. The caller catches this specific application type for each input, so one failure does not discard the next input. The throws declaration states which checked type may leave read; it does not raise an exception on the successful path. NumberFormatException belongs to the RuntimeException family, so it is unchecked. InvalidQuantityException directly extends Exception and is outside the RuntimeException and Error families, so it is checked. A conventional Java method that lets that checked type escape must declare it or instead catch it within the method. These categories describe compiler rules; they do not decide whether a failure is harmless.

```java
class InvalidQuantityException extends Exception {
    public InvalidQuantityException(String message, Throwable cause) {
        super(message, cause);
    }
}
class QuantityParser {
    public static int read(String text) throws InvalidQuantityException {
        int quantity;
        try {
            quantity = Integer.parseInt(text);
        } catch (NumberFormatException cause) {
            throw new InvalidQuantityException("Quantity must be a whole number.", cause);
        }
        if (quantity < 0) {
            throw new InvalidQuantityException("Quantity cannot be negative.", null);
        }
        return quantity;
    }
}
String[] inputs = {"8", "oops", "-3"};
for (String input : inputs) {
    try {
        System.out.println("Quantity: " + QuantityParser.read(input));
    } catch (InvalidQuantityException problem) {
        System.out.println("Problem: " + problem.getMessage());
        System.out.println("Has cause: " + (problem.getCause() != null));
    }
}
```

Expected output:

```text
Quantity: 8
Problem: Quantity must be a whole number.
Has cause: true
Problem: Quantity cannot be negative.
Has cause: false
```

Common error: Treating successful conversion as proof that the input meets the application rule. Assuming every custom exception must have a cause. Treating a throws declaration as a statement that always raises an exception.

</details>


### Pass a checked failure outward and inspect its cause

Predict the complete output before running each program. The first calls QuantityForm.read with "nine"; the second uses "6". QuantityForm.read declares the checked type that may continue outward to its caller. Locate the catch that ultimately handles it, and explain whether Validated and the helper return are reached on each path. getCause returns the attached earlier failure or null. Throwable is the common superclass used here to hold that failure reference; check for null before reading its message. Distinguish the application message from the original cause message. Then change only the helper-call argument to "-2", predict and run that complete program, and explain why the else branch is necessary. Explain why the helper needs its throws declaration in conventional Java source, while a method that may pass NumberFormatException outward has no corresponding compiler requirement. Keep all complete callers inside their supplied try/catch.

My predicted and actual output for "nine":

My predicted and actual output for "6":

My predicted and actual output for "-2":

Which method declares the checked failure and which catch receives it:

Skipped messages and returns after failure:

Application message versus original cause message:

Why checking for null matters:

Why throws does not raise a failure by itself:

Why NumberFormatException follows a different declaration rule:

My post-run explanation:


**First program:**


In [ ]:
class InvalidQuantityException extends Exception {
    public InvalidQuantityException(String message, Throwable cause) {
        super(message, cause);
    }
}
class QuantityParser {
    public static int read(String text) throws InvalidQuantityException {
        int quantity;
        try {
            quantity = Integer.parseInt(text);
        } catch (NumberFormatException cause) {
            throw new InvalidQuantityException("Quantity must be a whole number.", cause);
        }
        if (quantity < 0) {
            throw new InvalidQuantityException("Quantity cannot be negative.", null);
        }
        return quantity;
    }
}
class QuantityForm {
    public static int read(String text) throws InvalidQuantityException {
        System.out.println("Checking input");
        int value = QuantityParser.read(text);
        System.out.println("Validated");
        return value;
    }
}
try {
    System.out.println("Value: " + QuantityForm.read("nine"));
} catch (InvalidQuantityException problem) {
    System.out.println("Problem: " + problem.getMessage());
    Throwable cause = problem.getCause();
    if (cause != null) {
        System.out.println("Cause: " + cause.getMessage());
    } else {
        System.out.println("Cause: none");
    }
}
System.out.println("After request");


**Comparison program:**


In [ ]:
class InvalidQuantityException extends Exception {
    public InvalidQuantityException(String message, Throwable cause) {
        super(message, cause);
    }
}
class QuantityParser {
    public static int read(String text) throws InvalidQuantityException {
        int quantity;
        try {
            quantity = Integer.parseInt(text);
        } catch (NumberFormatException cause) {
            throw new InvalidQuantityException("Quantity must be a whole number.", cause);
        }
        if (quantity < 0) {
            throw new InvalidQuantityException("Quantity cannot be negative.", null);
        }
        return quantity;
    }
}
class QuantityForm {
    public static int read(String text) throws InvalidQuantityException {
        System.out.println("Checking input");
        int value = QuantityParser.read(text);
        System.out.println("Validated");
        return value;
    }
}
try {
    System.out.println("Value: " + QuantityForm.read("6"));
} catch (InvalidQuantityException problem) {
    System.out.println("Problem: " + problem.getMessage());
    Throwable cause = problem.getCause();
    if (cause != null) {
        System.out.println("Cause: " + cause.getMessage());
    } else {
        System.out.println("Cause: none");
    }
}
System.out.println("After request");


Record your own post-run explanation before opening the answer.

<details>
<summary>Show answer</summary>

QuantityForm.read prints Checking input, then calls QuantityParser.read. With "nine", parsing raises NumberFormatException, and the parser throws a new InvalidQuantityException that retains it. The checked application exception leaves QuantityForm.read through its declared throws contract; Validated and that method's return are skipped. The outer catch receives the application exception. getCause returns its original parsing exception, so the non-null branch safely reads its message. In the selected Workspace runtime that message is For input string: "nine". After request prints after handling. With "6", both methods return normally: Validated and Value: 6 print even though the helper declares throws. With "-2", conversion succeeds but the nonnegative rule fails; the attached cause is null, so the caller prints Cause: none without calling getMessage through null. The helper could instead handle the checked type itself; in this program it declares that the type may continue outward to its caller.

```java
class InvalidQuantityException extends Exception {
    public InvalidQuantityException(String message, Throwable cause) {
        super(message, cause);
    }
}
class QuantityParser {
    public static int read(String text) throws InvalidQuantityException {
        int quantity;
        try {
            quantity = Integer.parseInt(text);
        } catch (NumberFormatException cause) {
            throw new InvalidQuantityException("Quantity must be a whole number.", cause);
        }
        if (quantity < 0) {
            throw new InvalidQuantityException("Quantity cannot be negative.", null);
        }
        return quantity;
    }
}
class QuantityForm {
    public static int read(String text) throws InvalidQuantityException {
        System.out.println("Checking input");
        int value = QuantityParser.read(text);
        System.out.println("Validated");
        return value;
    }
}
try {
    System.out.println("Value: " + QuantityForm.read("nine"));
} catch (InvalidQuantityException problem) {
    System.out.println("Problem: " + problem.getMessage());
    Throwable cause = problem.getCause();
    if (cause != null) {
        System.out.println("Cause: " + cause.getMessage());
    } else {
        System.out.println("Cause: none");
    }
}
System.out.println("After request");
```

Expected output:

```text
Checking input
Problem: Quantity must be a whole number.
Cause: For input string: "nine"
After request
```

Common error: Removing the helper declaration without adding its own handler in conventional Java source. Reading a cause message before checking whether the cause is null. Expecting Validated after the parser has thrown the application exception.

**Check case 2.** Both parser and helper return normally. Declaring a possible failure does not raise it; the caller prints the returned value.

```java
class InvalidQuantityException extends Exception {
    public InvalidQuantityException(String message, Throwable cause) {
        super(message, cause);
    }
}
class QuantityParser {
    public static int read(String text) throws InvalidQuantityException {
        int quantity;
        try {
            quantity = Integer.parseInt(text);
        } catch (NumberFormatException cause) {
            throw new InvalidQuantityException("Quantity must be a whole number.", cause);
        }
        if (quantity < 0) {
            throw new InvalidQuantityException("Quantity cannot be negative.", null);
        }
        return quantity;
    }
}
class QuantityForm {
    public static int read(String text) throws InvalidQuantityException {
        System.out.println("Checking input");
        int value = QuantityParser.read(text);
        System.out.println("Validated");
        return value;
    }
}
try {
    System.out.println("Value: " + QuantityForm.read("6"));
} catch (InvalidQuantityException problem) {
    System.out.println("Problem: " + problem.getMessage());
    Throwable cause = problem.getCause();
    if (cause != null) {
        System.out.println("Cause: " + cause.getMessage());
    } else {
        System.out.println("Cause: none");
    }
}
System.out.println("After request");
```

Expected output:

```text
Checking input
Validated
Value: 6
After request
```

**Check case 3.** Parsing succeeds, so the range failure has no earlier exception attached. The null check selects Cause: none and prevents a method call through null.

```java
class InvalidQuantityException extends Exception {
    public InvalidQuantityException(String message, Throwable cause) {
        super(message, cause);
    }
}
class QuantityParser {
    public static int read(String text) throws InvalidQuantityException {
        int quantity;
        try {
            quantity = Integer.parseInt(text);
        } catch (NumberFormatException cause) {
            throw new InvalidQuantityException("Quantity must be a whole number.", cause);
        }
        if (quantity < 0) {
            throw new InvalidQuantityException("Quantity cannot be negative.", null);
        }
        return quantity;
    }
}
class QuantityForm {
    public static int read(String text) throws InvalidQuantityException {
        System.out.println("Checking input");
        int value = QuantityParser.read(text);
        System.out.println("Validated");
        return value;
    }
}
try {
    System.out.println("Value: " + QuantityForm.read("-2"));
} catch (InvalidQuantityException problem) {
    System.out.println("Problem: " + problem.getMessage());
    Throwable cause = problem.getCause();
    if (cause != null) {
        System.out.println("Cause: " + cause.getMessage());
    } else {
        System.out.println("Cause: none");
    }
}
System.out.println("After request");
```

Expected output:

```text
Checking input
Problem: Quantity cannot be negative.
Cause: none
After request
```

</details>


## Guided Practice

Complete these tasks in order. The intentionally empty code cells are safe to run, but remain unfinished until you write and check your code.


### Complete the checked exception path

The displayed draft is incomplete and for reading only. Copy it into the empty work cell. Replace CALL_PARENT, DECLARE_FAILURE and RAISE_FAILURE with `super`, `throws` and `throw`, each once. Keep every other statement unchanged. Predict all output lines, run the completed program, and explain the different jobs of the three replacements. Identify why the custom type is checked, where the caller handles it, and why the negative level has no earlier cause.

This sample is for repair:

```java
class InvalidLevelException extends Exception {
    public InvalidLevelException(String message, Throwable cause) {
        CALL_PARENT(message, cause);
    }
}
class LevelRules {
    public static int requireNonnegative(int level) DECLARE_FAILURE InvalidLevelException {
        if (level < 0) {
            RAISE_FAILURE new InvalidLevelException("Level cannot be negative.", null);
        }
        return level;
    }
}
int[] levels = {-2, 1};
for (int level : levels) {
    try {
        System.out.println("Level: " + LevelRules.requireNonnegative(level));
    } catch (InvalidLevelException problem) {
        System.out.println("Problem: " + problem.getMessage());
        System.out.println("Has cause: " + (problem.getCause() != null));
    }
}
```


My three replacements and each job:

My predicted complete output:

My actual complete output:

Why InvalidLevelException is checked:

Where the caller handles it:

Why the cause is null for the failed level:

My post-run explanation:

<details>
<summary>Show answer</summary>

CALL_PARENT is super: the exception constructor passes its message and cause to the Exception constructor. DECLARE_FAILURE is throws: requireNonnegative states that InvalidLevelException may reach its caller. RAISE_FAILURE is throw: the negative branch raises a new object now. The input -2 takes that branch, so the caller prints the problem and Has cause: false. There is no earlier exception attached because this is a direct range check. Input 1 reaches return, so Level: 1 prints. The specific catch meets the caller's obligation for this checked type.

```java
class InvalidLevelException extends Exception {
    public InvalidLevelException(String message, Throwable cause) {
        super(message, cause);
    }
}
class LevelRules {
    public static int requireNonnegative(int level) throws InvalidLevelException {
        if (level < 0) {
            throw new InvalidLevelException("Level cannot be negative.", null);
        }
        return level;
    }
}
int[] levels = {-2, 1};
for (int level : levels) {
    try {
        System.out.println("Level: " + LevelRules.requireNonnegative(level));
    } catch (InvalidLevelException problem) {
        System.out.println("Problem: " + problem.getMessage());
        System.out.println("Has cause: " + (problem.getCause() != null));
    }
}
```

Expected output:

```text
Problem: Level cannot be negative.
Has cause: false
Level: 1
```

Common error: Swapping throw and throws even though one is a statement and the other is part of a method declaration. Dropping the cause argument from the supplied constructor call. Placing the success print outside the protected method call.

</details>


### Trim text before conversion

Keep the exception class, range rule, handlers and output labels unchanged. Change only `Integer.parseInt(text)` to `Integer.parseInt(text.trim())`, using the String trimming you already know. First predict and run with {"8", "oops", "-3"}. Then change only the inputs initializer to {" 8 ", "oops", "-3"}, predict and run again. Finally test {"   "}. Explain what trimming changes, whether it makes every text a valid integer, and whether it changes the nonnegative rule or preservation of a parsing cause. Restore {"8", "oops", "-3"} after testing.


In [ ]:
class InvalidQuantityException extends Exception {
    public InvalidQuantityException(String message, Throwable cause) {
        super(message, cause);
    }
}
class QuantityParser {
    public static int read(String text) throws InvalidQuantityException {
        int quantity;
        try {
            quantity = Integer.parseInt(text);
        } catch (NumberFormatException cause) {
            throw new InvalidQuantityException("Quantity must be a whole number.", cause);
        }
        if (quantity < 0) {
            throw new InvalidQuantityException("Quantity cannot be negative.", null);
        }
        return quantity;
    }
}
String[] inputs = {"8", "oops", "-3"};
for (String input : inputs) {
    try {
        System.out.println("Quantity: " + QuantityParser.read(input));
    } catch (InvalidQuantityException problem) {
        System.out.println("Problem: " + problem.getMessage());
        System.out.println("Has cause: " + (problem.getCause() != null));
    }
}


My changed parse call:

My predicted and actual original-input output:

My predicted and actual padded-input output:

My predicted and actual whitespace-only output:

What trim changes and what it leaves unchanged:

Why a parsing cause still exists for whitespace-only input:

My post-run explanation and restored result:

<details>
<summary>Show answer</summary>

Trimming removes surrounding whitespace before conversion. The original inputs have no surrounding spaces, so this edit leaves their output unchanged. The padded " 8 " becomes "8" and converts successfully, while "oops" still fails conversion and "-3" still fails the nonnegative rule. A whitespace-only String becomes empty after trim, so Integer.parseInt still raises NumberFormatException and the custom exception retains it. The edit changes accepted text formatting; it does not change the application's numeric rule or the cause argument.

```java
class InvalidQuantityException extends Exception {
    public InvalidQuantityException(String message, Throwable cause) {
        super(message, cause);
    }
}
class QuantityParser {
    public static int read(String text) throws InvalidQuantityException {
        int quantity;
        try {
            quantity = Integer.parseInt(text.trim());
        } catch (NumberFormatException cause) {
            throw new InvalidQuantityException("Quantity must be a whole number.", cause);
        }
        if (quantity < 0) {
            throw new InvalidQuantityException("Quantity cannot be negative.", null);
        }
        return quantity;
    }
}
String[] inputs = {"8", "oops", "-3"};
for (String input : inputs) {
    try {
        System.out.println("Quantity: " + QuantityParser.read(input));
    } catch (InvalidQuantityException problem) {
        System.out.println("Problem: " + problem.getMessage());
        System.out.println("Has cause: " + (problem.getCause() != null));
    }
}
```

Expected output:

```text
Quantity: 8
Problem: Quantity must be a whole number.
Has cause: true
Problem: Quantity cannot be negative.
Has cause: false
```

Common error: Assuming trim converts words into numbers. Changing the range guard while editing input formatting. Removing the original cause when changing the parse call.

**Additional test: `Padded numeric input with other paths unchanged`.** Trimming the spaces makes the first input a valid 8. The nonnumeric and negative paths still preserve their original meanings and cause flags.

```java
class InvalidQuantityException extends Exception {
    public InvalidQuantityException(String message, Throwable cause) {
        super(message, cause);
    }
}
class QuantityParser {
    public static int read(String text) throws InvalidQuantityException {
        int quantity;
        try {
            quantity = Integer.parseInt(text.trim());
        } catch (NumberFormatException cause) {
            throw new InvalidQuantityException("Quantity must be a whole number.", cause);
        }
        if (quantity < 0) {
            throw new InvalidQuantityException("Quantity cannot be negative.", null);
        }
        return quantity;
    }
}
String[] inputs = {" 8 ", "oops", "-3"};
for (String input : inputs) {
    try {
        System.out.println("Quantity: " + QuantityParser.read(input));
    } catch (InvalidQuantityException problem) {
        System.out.println("Problem: " + problem.getMessage());
        System.out.println("Has cause: " + (problem.getCause() != null));
    }
}
```

Expected output:

```text
Quantity: 8
Problem: Quantity must be a whole number.
Has cause: true
Problem: Quantity cannot be negative.
Has cause: false
```

**Additional test: `Whitespace-only input`.** After trimming, no digits remain. Conversion still fails and its original NumberFormatException remains attached to the custom exception.

```java
class InvalidQuantityException extends Exception {
    public InvalidQuantityException(String message, Throwable cause) {
        super(message, cause);
    }
}
class QuantityParser {
    public static int read(String text) throws InvalidQuantityException {
        int quantity;
        try {
            quantity = Integer.parseInt(text.trim());
        } catch (NumberFormatException cause) {
            throw new InvalidQuantityException("Quantity must be a whole number.", cause);
        }
        if (quantity < 0) {
            throw new InvalidQuantityException("Quantity cannot be negative.", null);
        }
        return quantity;
    }
}
String[] inputs = {"   "};
for (String input : inputs) {
    try {
        System.out.println("Quantity: " + QuantityParser.read(input));
    } catch (InvalidQuantityException problem) {
        System.out.println("Problem: " + problem.getMessage());
        System.out.println("Has cause: " + (problem.getCause() != null));
    }
}
```

Expected output:

```text
Problem: Quantity must be a whole number.
Has cause: true
```

</details>


### Repair a lost original cause

The displayed draft still prints a clear message for "oops", but the conversion catch passes null to the custom exception constructor. Predict the complete output and identify which report violates the cause-preservation requirement. Keep the faulty draft in Markdown. Copy the full program into the empty work cell and repair only that constructor argument. Preserve the messages, range check, inputs and caller. Predict and run the repair, then explain why the successful value and application message alone could not reveal this defect. Explain why the negative-input branch should still pass null.

This sample is for repair:

```java
class InvalidQuantityException extends Exception {
    public InvalidQuantityException(String message, Throwable cause) {
        super(message, cause);
    }
}
class QuantityParser {
    public static int read(String text) throws InvalidQuantityException {
        int quantity;
        try {
            quantity = Integer.parseInt(text);
        } catch (NumberFormatException cause) {
            throw new InvalidQuantityException("Quantity must be a whole number.", null);
        }
        if (quantity < 0) {
            throw new InvalidQuantityException("Quantity cannot be negative.", null);
        }
        return quantity;
    }
}
String[] inputs = {"8", "oops", "-3"};
for (String input : inputs) {
    try {
        System.out.println("Quantity: " + QuantityParser.read(input));
    } catch (InvalidQuantityException problem) {
        System.out.println("Problem: " + problem.getMessage());
        System.out.println("Has cause: " + (problem.getCause() != null));
    }
}
```


My predicted faulty output:

The report that violates the contract:

My repaired constructor argument:

My predicted complete repaired output:

My actual repaired output:

Why the same message can hide a missing cause:

Why the negative-input cause stays null:

My post-run explanation:

<details>
<summary>Show answer</summary>

The text "8" converts to 8 and passes the nonnegative rule, so Quantity: 8 prints. "oops" cannot convert: Integer.parseInt raises NumberFormatException. The inner catch constructs and throws InvalidQuantityException with the application message and that original object as its cause. The outer catch prints the message and Has cause: true. "-3" converts successfully, then fails the quantity < 0 guard. The new InvalidQuantityException has a null cause because there was no earlier conversion failure; the outer catch prints Has cause: false. The caller catches this specific application type for each input, so one failure does not discard the next input. The throws declaration states which checked type may leave read; it does not raise an exception on the successful path. In the faulty conversion catch, passing null discards the original NumberFormatException even though the application message still describes the conversion failure. The repaired constructor call passes cause, the actual caught object. The direct negative-value rejection correctly keeps null because conversion succeeded. Checking the cause flag distinguishes the faulty and repaired programs; checking only the visible problem sentence would miss the defect.

```java
class InvalidQuantityException extends Exception {
    public InvalidQuantityException(String message, Throwable cause) {
        super(message, cause);
    }
}
class QuantityParser {
    public static int read(String text) throws InvalidQuantityException {
        int quantity;
        try {
            quantity = Integer.parseInt(text);
        } catch (NumberFormatException cause) {
            throw new InvalidQuantityException("Quantity must be a whole number.", cause);
        }
        if (quantity < 0) {
            throw new InvalidQuantityException("Quantity cannot be negative.", null);
        }
        return quantity;
    }
}
String[] inputs = {"8", "oops", "-3"};
for (String input : inputs) {
    try {
        System.out.println("Quantity: " + QuantityParser.read(input));
    } catch (InvalidQuantityException problem) {
        System.out.println("Problem: " + problem.getMessage());
        System.out.println("Has cause: " + (problem.getCause() != null));
    }
}
```

Expected output:

```text
Quantity: 8
Problem: Quantity must be a whole number.
Has cause: true
Problem: Quantity cannot be negative.
Has cause: false
```

Common error: Replacing the conversion message without restoring its original cause. Changing both null arguments even though the range failure has no earlier exception. Treating the cause flag as the success result of parsing.

</details>


## Independent Practice

### Build a checked seat-count parser

Write the complete exception class, parser and caller in the work cell. Create `InvalidSeatCountException extends Exception` with one constructor taking `String message` and `Throwable cause`, and pass both to `super`. Create `SeatParser` with a `public static int read(String text)` method that declares `throws InvalidSeatCountException`. Parse with `Integer.parseInt(text)`. If conversion raises `NumberFormatException`, throw a new `InvalidSeatCountException` with message `Seats must be a whole number.` and the original cause. Reject a parsed count below one with message `Seats must be positive.` and null cause; otherwise return the count. Process `String[] inputs = {"4", "many", "0"};` with a specific `InvalidSeatCountException` handler inside each iteration. On success print `Seats: ` plus the result. On failure print `Problem: ` plus the message, then `Has cause: ` plus whether `getCause()` is non-null. Keep complete callers inside try/catch or declare the checked exception from their methods so they also follow conventional Java rules. Predict all output lines before running. Explain why zero fails this task although it is valid for the quantity parser, and why nonnumeric input and zero have different cause flags. Test one, a negative value and an integer too large for int in the next stage.

My exception class and parser contract:

My predicted complete output:

My actual complete output:

Why zero differs from the quantity example:

The cause on each failed path and why:

Where the checked failure is handled or declared:

My post-run explanation:


### Check the positive boundary and conversion limits

Use your complete unchanged SeatParser program for each row. Change only the inputs initializer. Predict all output lines, run, and record the actual message and cause flag or successful value. Test {"1"}, {"-2"}, {"2147483648"} and {" 1 "} as well as the baseline. The largest int value is 2147483647; explain why digits can still fail conversion when they represent a larger value. Explain which test checks the smallest valid seat count, which reaches range validation after successful conversion, and which fail before that guard. The fixed SeatParser uses Integer.parseInt(text) without the trimming change from guided practice. Repair any mismatch, repeat the affected cases and restore {"4", "many", "0"}. Retain a post-run explanation of why both the message and cause flag matter.

| inputs | My predicted output | My actual output | Conversion or range path |
|---|---|---|---|
| {"4", "many", "0"} | | | |
| {"1"} | | | |
| {"-2"} | | | |
| {"2147483648"} | | | |
| {" 1 "} | | | |


Why 1 is the lower valid boundary:

Why a sequence of digits can fail conversion:

Which cases reach the application range guard:

Which cases preserve an earlier exception and why:

My correction and repeated checks:

My actual restored baseline output:

My post-run explanation of message and cause checks:


<details>
<summary>Show answer</summary>

The custom type directly extends Exception and is checked. Its one constructor preserves both message and cause. SeatParser.read declares that type, wraps NumberFormatException without losing the original object, and separately rejects counts below one. The text "4" returns 4. "many" fails conversion, so the application message is accompanied by Has cause: true. "0" converts successfully but fails the positive-count rule, so its cause is null and Has cause: false prints. Each iteration has a specific handler, allowing every input to receive its report. A conventional method that lets the checked type continue outward must declare it; otherwise it needs its own handler. The value 1 checks the smallest accepted seat count. The negative value -2 parses but fails the application rule with no earlier cause. The value 2147483648 is outside the int range and cannot be converted, so its original NumberFormatException is preserved. The surrounding spaces in " 1 " also prevent conversion in this unchanged parser. Its numeric guard is never reached on either conversion failure. The tests distinguish the two failure paths and would expose a parser that loses the original cause or incorrectly accepts zero.

```java
class InvalidSeatCountException extends Exception {
    public InvalidSeatCountException(String message, Throwable cause) {
        super(message, cause);
    }
}
class SeatParser {
    public static int read(String text) throws InvalidSeatCountException {
        int seats;
        try {
            seats = Integer.parseInt(text);
        } catch (NumberFormatException cause) {
            throw new InvalidSeatCountException("Seats must be a whole number.", cause);
        }
        if (seats < 1) {
            throw new InvalidSeatCountException("Seats must be positive.", null);
        }
        return seats;
    }
}
String[] inputs = {"4", "many", "0"};
for (String input : inputs) {
    try {
        System.out.println("Seats: " + SeatParser.read(input));
    } catch (InvalidSeatCountException problem) {
        System.out.println("Problem: " + problem.getMessage());
        System.out.println("Has cause: " + (problem.getCause() != null));
    }
}
```

Expected output:

```text
Seats: 4
Problem: Seats must be a whole number.
Has cause: true
Problem: Seats must be positive.
Has cause: false
```

Common error: Reusing the quantity < 0 guard when seat counts must be at least one. Passing null when wrapping the original NumberFormatException. Attaching an invented parsing failure when conversion actually succeeded. Leaving a checked failure neither handled nor declared in conventional Java methods. Changing the exact method names, messages or caller inputs.

**Additional test: {"1"}.** One is the smallest valid seat count. Conversion succeeds and the positive-count rule accepts it, so there is no exception report.

```java
class InvalidSeatCountException extends Exception {
    public InvalidSeatCountException(String message, Throwable cause) {
        super(message, cause);
    }
}
class SeatParser {
    public static int read(String text) throws InvalidSeatCountException {
        int seats;
        try {
            seats = Integer.parseInt(text);
        } catch (NumberFormatException cause) {
            throw new InvalidSeatCountException("Seats must be a whole number.", cause);
        }
        if (seats < 1) {
            throw new InvalidSeatCountException("Seats must be positive.", null);
        }
        return seats;
    }
}
String[] inputs = {"1"};
for (String input : inputs) {
    try {
        System.out.println("Seats: " + SeatParser.read(input));
    } catch (InvalidSeatCountException problem) {
        System.out.println("Problem: " + problem.getMessage());
        System.out.println("Has cause: " + (problem.getCause() != null));
    }
}
```

Expected output:

```text
Seats: 1
```

**Additional test: {"-2"}.** The text converts to a negative integer, then fails the positive-count rule. No earlier parsing exception exists, so the cause is null.

```java
class InvalidSeatCountException extends Exception {
    public InvalidSeatCountException(String message, Throwable cause) {
        super(message, cause);
    }
}
class SeatParser {
    public static int read(String text) throws InvalidSeatCountException {
        int seats;
        try {
            seats = Integer.parseInt(text);
        } catch (NumberFormatException cause) {
            throw new InvalidSeatCountException("Seats must be a whole number.", cause);
        }
        if (seats < 1) {
            throw new InvalidSeatCountException("Seats must be positive.", null);
        }
        return seats;
    }
}
String[] inputs = {"-2"};
for (String input : inputs) {
    try {
        System.out.println("Seats: " + SeatParser.read(input));
    } catch (InvalidSeatCountException problem) {
        System.out.println("Problem: " + problem.getMessage());
        System.out.println("Has cause: " + (problem.getCause() != null));
    }
}
```

Expected output:

```text
Problem: Seats must be positive.
Has cause: false
```

**Additional test: {"2147483648"}.** These digits represent a value larger than the maximum int value, 2147483647. Integer.parseInt cannot produce an int and raises NumberFormatException. The application exception retains that cause; the range check is not reached.

```java
class InvalidSeatCountException extends Exception {
    public InvalidSeatCountException(String message, Throwable cause) {
        super(message, cause);
    }
}
class SeatParser {
    public static int read(String text) throws InvalidSeatCountException {
        int seats;
        try {
            seats = Integer.parseInt(text);
        } catch (NumberFormatException cause) {
            throw new InvalidSeatCountException("Seats must be a whole number.", cause);
        }
        if (seats < 1) {
            throw new InvalidSeatCountException("Seats must be positive.", null);
        }
        return seats;
    }
}
String[] inputs = {"2147483648"};
for (String input : inputs) {
    try {
        System.out.println("Seats: " + SeatParser.read(input));
    } catch (InvalidSeatCountException problem) {
        System.out.println("Problem: " + problem.getMessage());
        System.out.println("Has cause: " + (problem.getCause() != null));
    }
}
```

Expected output:

```text
Problem: Seats must be a whole number.
Has cause: true
```

**Additional test: {" 1 "}.** The fixed SeatParser uses Integer.parseInt without trimming. Surrounding spaces prevent conversion, even though the central digit is positive. The original NumberFormatException is retained as the cause.

```java
class InvalidSeatCountException extends Exception {
    public InvalidSeatCountException(String message, Throwable cause) {
        super(message, cause);
    }
}
class SeatParser {
    public static int read(String text) throws InvalidSeatCountException {
        int seats;
        try {
            seats = Integer.parseInt(text);
        } catch (NumberFormatException cause) {
            throw new InvalidSeatCountException("Seats must be a whole number.", cause);
        }
        if (seats < 1) {
            throw new InvalidSeatCountException("Seats must be positive.", null);
        }
        return seats;
    }
}
String[] inputs = {" 1 "};
for (String input : inputs) {
    try {
        System.out.println("Seats: " + SeatParser.read(input));
    } catch (InvalidSeatCountException problem) {
        System.out.println("Problem: " + problem.getMessage());
        System.out.println("Has cause: " + (problem.getCause() != null));
    }
}
```

Expected output:

```text
Problem: Seats must be a whole number.
Has cause: true
```

</details>


## Summary

A checked exception creates a catch-or-declare obligation in conventional Java source. An unchecked exception does not create that obligation. A throw statement raises an exception now, while a throws declaration states a possible escaping failure. A custom type names an application failure, and its cause preserves an earlier failure.

Close the answers. Explain why -1 can pass conversion and still fail validation. Distinguish a message from a cause.


## Reflection

A registration system accepts a number of seats. Describe a conversion failure, a valid integer that violates the application rule, and the information each exception should preserve.

**My design and explanation:**

Next, you will ensure that a failure does not prevent resource cleanup.


## Supplemental Reading

- [Throwing Java exceptions](https://dev.java/learn/exceptions/throwing/) covers throw, method declarations, and custom types.
- [Java 21 exception checking](https://docs.oracle.com/javase/specs/jls/se21/html/jls-11.html) states the checked and unchecked rules.
- [Java 21 Exception API](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/lang/Exception.html) documents constructors that retain a message and cause.
- [Java 21 Integer.parseInt](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/lang/Integer.html#parseInt(java.lang.String)) specifies decimal conversion and failure cases.
